# Your own Colab copy

Choose File, Save a copy in Drive before working. Run the setup cell once, then work through the starter. Download your notebook and exported ZIP before disconnecting. Colab files can disappear when a runtime ends. Submit through the hub after finishing your attempt.

In [ ]:
from pathlib import Path
import hashlib
import os
import urllib.request
import zipfile

work = Path('/content/icaic-catacombs') if Path('/content').exists() else Path.cwd() / 'icaic-catacombs'
work.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')
os.environ.setdefault('SDL_AUDIODRIVER', 'dummy')
os.environ.setdefault('PYGAME_HIDE_SUPPORT_PROMPT', '1')
payload = urllib.request.urlopen('https://elteomc.github.io/icaic-practice-hub/packs/catacombs.zip', timeout=90).read()
assert hashlib.sha256(payload).hexdigest() == '5cdeb23c0be0bf34fb5baaf42d2b9e5bdfb0d1949bd76ab0413981afae4fd61d', 'The task pack changed. Reopen the latest notebook from the hub.'
archive_path = work / 'task.zip'
archive_path.write_bytes(payload)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(work)
os.chdir(work / 'catacombs')
import sys
sys.path.insert(0, str(Path.cwd()))
print('Task folder:', Path.cwd())
try:
    import creepy_catacombs_s1
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'creepy-catacombs-s1==0.1.4'])


# Catacomb Adventure

All packages are already installed. Start by reading the map and one transition. The initial exported policy is deliberately weak. The full original exercise is in Source_Reading.ipynb. Read Statement.md before submitting.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import creepy_catacombs_s1

env = gym.make('CreepyCatacombs-v0', render_mode='rgb_array')
state, info = env.reset(seed=2025)
print('Observation:', state, 'actions:', env.action_space.n)
print('Start:', info['agent_pos'], 'goal:', info['goal_pos'])
print(info['grid'])
plt.imshow(env.render())
plt.axis('off')
plt.show()

## 1. Inspect one transition

Actions are 0 up, 1 right, 2 down, 3 left. Reset before testing another route.

In [ ]:
state, info = env.reset(seed=2025)
next_state, reward, terminated, truncated, info = env.step(0)
print('Next state:', next_state, 'reward:', reward)
print('Terminated:', terminated, 'truncated:', truncated)

## 2. Establish a route

Replace the sequence with your safe route. Walls are -1, pits are -2, the start is 1, and the exit is 2. The source notebook guides a breadth-first search if you prefer to compute it.

In [ ]:
actions = [0, 0]
state, info = env.reset(seed=2025)
total_reward = 0
for action in actions:
    state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break
print('Route reward:', total_reward, 'position:', info['agent_pos'])

## 3. Learn action values

Implement Monte Carlo control and SARSA in the cells below. Begin with zero-filled tables of shape 56 by 4, use exploratory episodes while training, and log episode returns. For Monte Carlo, update using complete returns. For SARSA, update from reward and the next chosen action value. Reset to seed 2025 for this fixed-map pilot. You can use the original notebook's unfinished training loops as scaffolding.

In [ ]:
mc_q = np.zeros((env.observation_space.n, env.action_space.n))
sarsa_q = np.zeros_like(mc_q)
mc_returns = []
sarsa_returns = []

# Implement your training loops here


## 4. Export your submission

The export cell already runs. Improve the route and tables, rerun it, then upload policy.json. Keep your notebook and plots for peer review.

In [ ]:
payload = {'actions': [int(a) for a in actions], 'mc_q': mc_q.tolist(), 'sarsa_q': sarsa_q.tolist()}
Path('policy.json').write_text(json.dumps(payload, allow_nan=False))
print('Saved policy.json')
print('Validation tie seeds: 7, 11, 19, 23, 31')

## 5. Review

Plot raw episode returns and a moving average after training. Explain one changed state-action value and compare the two learning rules. Then attempt the optional source exercises with zombies. Automatic points cover the fixed map only.

In [ ]:
env.close()

## Download for the hub

Run this export cell to download your ZIP. Open Submit & notes on the task page and follow the available grading instructions. If you are practicing independently, finish your attempt before checking the final practice score.

In [ ]:
from pathlib import Path
import zipfile
assert Path('policy.json').is_file(), 'Save policy.json first'
with zipfile.ZipFile('hub-upload.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write('policy.json', 'policy.json')
output = 'hub-upload.zip'
try:
    from google.colab import files
    files.download(output)
except ImportError:
    print('Upload this file:', Path(output).resolve())
